In [1]:
#| default_exp restxl

In [132]:
#| hide
import nbdev; nbdev.nbdev_export()

In [15]:
!ln -s src rest/src

ln: failed to create symbolic link 'rest/src': File exists


In [16]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [17]:
#| export
from rest.core import init_instance, generate
singleton, model_path = init_instance()

In [19]:
#| export
seq_length = 512

from src.xl_wrapper import RuGPT3XL
model = RuGPT3XL.from_pretrained(
    "sberbank-ai/rugpt3xl",
    weights_path=f"./models/xl/{model_path}.model",
    deepspeed_config_path="src/deepspeed_config/gpt3_xl_sparse_2048.json",
    seq_len=seq_length
)
tokenizer = model.tokenizer
model.cuda()
model.eval();

The default process group has already initialized...
Use alternating sparse & dense attention layers


In [21]:
sum(p.numel() for p in model.parameters())

1315737600

In [22]:
#| export
import deepspeed, torch
ds_engine = deepspeed.init_inference(model,
                                 mp_size=1,
                                 dtype=torch.half,
                                 checkpoint=None,
                                 replace_method='auto',
                                 replace_with_kernel_inject=True)
model = ds_engine.module

[2022-11-11 19:26:54,339] [INFO] [logging.py:68:log_dist] [Rank -1] DeepSpeed info: version=0.7.5+28d4fdb, git-hash=28d4fdb, git-branch=master
[2022-11-11 19:26:54,340] [INFO] [logging.py:68:log_dist] [Rank -1] quantize_bits = 8 mlp_extra_grouping = False, quantize_groups = 1


In [12]:
#| export
def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool):
    return generate(model, tokenizer, seq_length, prompt, length, num_samples, allow_linebreak)

In [13]:
%%time
get_sample(' - ты кто? \n - ', 50, 4, False)

not setting adaptive thresholding
CPU times: user 26 s, sys: 867 ms, total: 26.8 s
Wall time: 15.9 s


['.........ответ. И тут скорее б возрадовался кто-нибудь, чем над этим задумался. Ненужный дятел, взявши трещину на стволе, дерево навестил. Но лес вздохнул с завистью.',
 '............. - Ты кто? Я-бля............................ - Ты бля?............................. - А ты? Ты кто? (непрерывно и много раз) - Я... А ты?...........................',
 '.................автоответчик, сто семнадцать. Вы слушаете сто семнадцать?............. Отвечайте, я вам...........! (с) "Властелин колец", ВК. Я отвечаю. -Да, слушаю.',
 '........... Сова! Со мной не здоровайся! Скажи ей, что это я - птица, на которую весной падает сон! Со мной нельзя здороваться! Ты меня не знаешь! Я - птица, которую Соня... Хочешь, я тебе расскажу?']

In [14]:
%%time
print(get_sample(' - ты кто?', 50, 1, True)[0])

not setting adaptive thresholding

Душа болит за Родину, брат!
Уж сколько зим с тех пор минуло -
Бушует классовая война.
Еще тогда мы в школе были,
Чтоб сметь учиться на отлично.

CPU times: user 1min 15s, sys: 2.05 s, total: 1min 17s
Wall time: 3.7 s
